# CellHit upload preparation

Prepare raw TCGA count matrices for patients without precomputed CellHit predictions.

In [ ]:
import os
from pathlib import Path

project_root = os.environ.get("MSS_TMB_PROJECT_ROOT")
if project_root:
    PROJECT_ROOT = Path(project_root).expanduser().resolve()
else:
    cwd = Path.cwd()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "scripts").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Set MSS_TMB_PROJECT_ROOT to the project directory.")


from pathlib import Path
import math, re, warnings
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = PROJECT_ROOT / 'data'

COUNT_FILES = {
    'COAD': BASE_DIR / 'coad' / 'count.csv',
    'READ': BASE_DIR / 'read' / 'count.csv',
    'UCEC': BASE_DIR / 'UCEC' / 'count_gene_name.csv',
}

CRC_CLINICAL = BASE_DIR / 'clinical_data.csv'
UCEC_CLINICAL = BASE_DIR / 'ucec_clinical.csv'
OUTPUT_DIR = BASE_DIR / 'CellHit' / 'cellhit_uploads'
CELLHIT_DIR = BASE_DIR / 'CellHit'
CELLHIT_SOURCES = {
    'GDSC': CELLHIT_DIR / 'full_results_gdsc.csv',
    'PRISM': CELLHIT_DIR / 'full_results_prism.csv',
}
PRIOR_PREDICTIONS_DB = CELLHIT_DIR / 'CRC_UCEC_MSI_TMB_RANKINGS' / 'cellhit_matched_patients.sqlite'
PRIOR_CACHE_MANIFEST = CELLHIT_DIR / 'CRC_UCEC_MSI_TMB_RANKINGS' / 'cache_manifest.json'
PRESENCE_CACHE_FILE = OUTPUT_DIR / 'cellhit_present_patients_cache.json'

MISSING_MODE = 'both'
REBUILD_PATIENT_PRESENCE_CACHE = False
PRESENCE_SCAN_CHUNK_SIZE = 300_000

TISSUE_BY_CODE = {'COAD': 'Bowel', 'READ': 'Bowel', 'UCEC': 'Uterus'}

MAX_SAMPLES_PER_UPLOAD = 150
MIN_SAMPLES_PER_UPLOAD = 3
MIN_GENES = 10000
COUNT_READ_CHUNK = 500
ALLOW_FRACTIONAL_COUNTS = False
REQUIRE_EXPLICIT_TUMOR_SAMPLE_TYPE = False
LAYOUT_OVERRIDE = {'COAD': 'auto', 'READ': 'auto', 'UCEC': 'auto'}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for code_, path in COUNT_FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f'{code_}: missing count matrix: {path}')
    if any(w in path.name.lower() for w in ['tpm', 'fpkm', 'tmm', 'celligner', 'log']):
        raise ValueError(f'{code_}: must select a raw counts file, not {path}')
    print(f'{code_}: {path} ({path.stat().st_size / 1024**2:.1f} MiB)')
for source, p in CELLHIT_SOURCES.items():
    if not p.is_file():
        raise FileNotFoundError(f'Missing {source} CellHit predictions at {p}; do not guess coverage')
    print(f'{source} precomputed results: {p.stat().st_size / 1024**3:.2f} GiB')
for p in [CRC_CLINICAL, UCEC_CLINICAL]:
    if not p.is_file():
        raise FileNotFoundError(f'Missing clinical file: {p}')
print('Output:', OUTPUT_DIR)

## Read clinical MSI/TMB labels

Match predictions to clinical groups by TCGA patient barcode.

In [ ]:
TCGA_RE = re.compile(r'(TCGA[-.][A-Z0-9]{2}[-.][A-Z0-9]{4})', re.I)
VALID_GROUPS = {'MSI-H','MSS/TMB-H','MSS/TMB-L'}

def normalize_barcode(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper().replace('.', '-')
    if s.startswith('TCGA_'):
        s = s.replace('_', '-')
    m = TCGA_RE.search(s)
    return s[m.start():] if m else None

def patient_id(x):
    b = normalize_barcode(x)
    return b[:12] if b else None

def sample_type(x):
    b = normalize_barcode(x)
    if not b:
        return None
    segments = b.split('-')
    if len(segments) < 4:
        return None
    part = segments[3][:2]
    return int(part) if part.isdigit() else None

def load_clinical(path, cohort):
    names = pd.read_csv(path, nrows=0).columns.tolist()
    keep = [x for x in ['Tumor_Sample_Barcode', 'sample_submitter_id','case_submitter_id',
                         'dataset','project_id','MSI_TMB'] if x in names]
    if 'MSI_TMB' not in keep:
        raise ValueError(f'{path}: missing MSI_TMB group labels')
    d = pd.read_csv(path, usecols=keep, low_memory=False)
    idcol = next((x for x in ['Tumor_Sample_Barcode','sample_submitter_id','case_submitter_id'] if x in d), None)
    if idcol is None:
        raise ValueError(f'{path}: missing TCGA barcode columns')
    if cohort == 'CRC':
        if 'dataset' not in d:
            raise ValueError('CRC clinical_data.csv must include dataset for COAD/READ')
        d['TCGA_CODE'] = d['dataset'].astype('string').str.upper().str.replace('TCGA-','',regex=False)
        d = d[d.TCGA_CODE.isin(['COAD','READ'])].copy()
    else:
        d['TCGA_CODE'] = 'UCEC'
        if 'project_id' in d:
            d = d[d['project_id'].astype('string').str.upper().eq('TCGA-UCEC')].copy()
    d['clinical_sample'] = d[idcol].map(normalize_barcode)
    d['patient_id'] = d['clinical_sample'].map(patient_id)
    d['group_label'] = d['MSI_TMB'].astype('string').str.upper().str.strip()
    d = d[d.group_label.isin(VALID_GROUPS)].dropna(subset=['patient_id'])
    conflict = d.groupby('patient_id').agg(n_group=('group_label','nunique'), n_tissue=('TCGA_CODE','nunique'))
    conflict = conflict[(conflict.n_group > 1) | (conflict.n_tissue > 1)]
    if len(conflict):
        raise ValueError(f'Conflicting clinical labels for {cohort}: {conflict.index.tolist()[:5]}')
    d = d.sort_values(['patient_id','clinical_sample']).drop_duplicates('patient_id')
    d['cancer'] = cohort
    return d[['patient_id','clinical_sample','TCGA_CODE','cancer','group_label']]

clinical = pd.concat([
    load_clinical(CRC_CLINICAL,'CRC'),
    load_clinical(UCEC_CLINICAL,'UCEC')], ignore_index=True)
if clinical.patient_id.duplicated().any():
    raise ValueError('A patient occurs in multiple cohorts in the clinical files')
clinical.to_csv(OUTPUT_DIR/'clinical_patients_reference.csv',index=False)
display(clinical.groupby(['TCGA_CODE','group_label']).size().rename('clinical_n').reset_index())

## Identify patients without published predictions

In [ ]:
import json, sqlite3, time

def source_fingerprint(path):
    s = path.stat()
    return {'path': str(path.resolve()), 'size': s.st_size, 'mtime_ns': s.st_mtime_ns}

current_fingerprints = {key: source_fingerprint(path) for key, path in CELLHIT_SOURCES.items()}

def read_prior_sqlite_patient_sets():
    if not PRIOR_PREDICTIONS_DB.is_file() or not PRIOR_CACHE_MANIFEST.is_file():
        return None
    try:
        saved = json.loads(PRIOR_CACHE_MANIFEST.read_text())
        old_fps = saved.get('sources', {})
        if any(old_fps.get(k) != current_fingerprints[k] for k in CELLHIT_SOURCES):
            print('Previous SQLite manifest does not match source CSVs; do not use it')
            return None
        with sqlite3.connect(PRIOR_PREDICTIONS_DB) as con:
            tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
            if 'patient_predictions' not in tables:
                return None
            rows = con.execute('SELECT DISTINCT dataset, patient_id FROM patient_predictions').fetchall()
        present = {k: set() for k in CELLHIT_SOURCES}
        for ds, pid in rows:
            if ds in present and patient_id(pid):
                present[ds].add(patient_id(pid))
        print('Reused previously validated CellHit SQLite cache for patient presence')
        return present
    except (OSError, ValueError, sqlite3.Error, TypeError) as exc:
        print('Prior SQLite cache cannot be used:', exc)
        return None

def scan_cellhit_patient_sets():
    present = {}
    for dataset, fp in CELLHIT_SOURCES.items():
        head = pd.read_csv(fp, nrows=0).columns
        if 'SampleIndex' not in head or 'Source' not in head:
            raise ValueError(f'{fp.name}: Source and SampleIndex columns required')
        ids=set(); scanned=0
        start=time.time()
        for part in pd.read_csv(fp, usecols=['Source', 'SampleIndex'],
                                dtype={'Source':'string','SampleIndex':'string'},
                                chunksize=PRESENCE_SCAN_CHUNK_SIZE):
            scanned += len(part)
            keep = part['Source'].str.upper().eq('TCGA').fillna(False) & \
                   part['SampleIndex'].str.upper().str.startswith('TCGA-').fillna(False)
            ids.update(pid for pid in part.loc[keep, 'SampleIndex'].map(patient_id) if pid)
            if scanned % (10 * PRESENCE_SCAN_CHUNK_SIZE) == 0:
                print(f'  {dataset}: {scanned:,} rows scanned, {len(ids):,} TCGA patients')
        present[dataset] = ids
        print(f'{dataset}: {len(ids):,} unique TCGA patients from {scanned:,} rows in '
              f'{time.time()-start:.1f}s')
    return present

present = None
if not REBUILD_PATIENT_PRESENCE_CACHE and PRESENCE_CACHE_FILE.is_file():
    try:
        item = json.loads(PRESENCE_CACHE_FILE.read_text())
        if item.get('fingerprints') == current_fingerprints:
            present = {k: set(item['present'][k]) for k in CELLHIT_SOURCES}
            print('Loaded compact patient-presence cache (no large-file scan)')
    except (ValueError, OSError, KeyError) as exc:
        print('Ignoring invalid patient-presence cache:', exc)
if present is None and not REBUILD_PATIENT_PRESENCE_CACHE:
    present = read_prior_sqlite_patient_sets()
if present is None:
    print('Scanning CellHit files for predicted patient IDs (one-time operation)...')
    present = scan_cellhit_patient_sets()
PRESENCE_CACHE_FILE.write_text(json.dumps({
    'fingerprints': current_fingerprints,
    'present': {k: sorted(v) for k,v in present.items()},
}, indent=2))

clinical_full = clinical.copy()
status = clinical_full.copy()
status['has_GDSC'] = status['patient_id'].isin(present['GDSC'])
status['has_PRISM'] = status['patient_id'].isin(present['PRISM'])
status['missing_both'] = ~status['has_GDSC'] & ~status['has_PRISM']
status['missing_either'] = ~status['has_GDSC'] | ~status['has_PRISM']
status.to_csv(OUTPUT_DIR / 'all_clinical_patient_cellhit_status.csv',index=False)

if MISSING_MODE not in {'both','either'}:
    raise ValueError("MISSING_MODE must be 'both' or 'either'")
missing_col = 'missing_both' if MISSING_MODE == 'both' else 'missing_either'
missing_status = status.loc[status[missing_col]].copy()
missing_status.to_csv(OUTPUT_DIR / 'patients_to_upload_missing_cellhit.csv',index=False)
status.loc[~status[missing_col]].to_csv(OUTPUT_DIR / 'patients_already_covered.csv',index=False)
clinical = clinical_full.loc[clinical_full.patient_id.isin(missing_status.patient_id)].copy()

summary = status.groupby(['TCGA_CODE','group_label'], as_index=False).agg(
    n_clinical=('patient_id','size'),
    n_present_GDSC=('has_GDSC','sum'),
    n_present_PRISM=('has_PRISM','sum'),
    n_missing_both=('missing_both','sum'),
    n_missing_either=('missing_either','sum'),
)
summary['n_selected_to_upload'] = summary['n_missing_both'] if MISSING_MODE == 'both' else summary['n_missing_either']
summary.to_csv(OUTPUT_DIR / 'cellhit_existing_vs_missing_by_group.csv', index=False)
print(f'Missing policy: {MISSING_MODE}. Already downloaded -> no re-export')
print('Patients requiring new predictions:',len(clinical),'/',len(clinical_full))
display(summary)
if clinical.empty:
    raise RuntimeError('All clinical patients are already covered under MISSING_MODE; no uploads needed')

## Inspect count matrices

In [ ]:
def separator_for(path):
    name = str(path).lower().removesuffix('.gz')
    return '\t' if name.endswith(('.tsv','.txt')) else ','

def detect_layout(path, override='auto'):
    if override != 'auto':
        if override not in {'genes_rows','samples_rows'}:
            raise ValueError('Layout must be auto, genes_rows, or samples_rows')
        return override
    sep = separator_for(path)
    header = pd.read_csv(path, sep=sep, nrows=0).columns.tolist()
    if len(header) < 3:
        raise ValueError(f'Expected at least 3 columns in {path}')
    sample_headers = sum(patient_id(x) is not None for x in header[1:])
    first_values = pd.read_csv(path, sep=sep, usecols=[header[0]], nrows=8).iloc[:,0].tolist()
    sample_rows = sum(patient_id(v) is not None for v in first_values)
    if sample_headers > 0 and sample_rows == 0:
        return 'genes_rows'
    if sample_rows > 0 and sample_headers == 0:
        return 'samples_rows'
    raise ValueError(f'Cannot confidently detect {path.name}: {sample_headers} sample headers, '
                     f'{sample_rows} sample ID values. Set LAYOUT_OVERRIDE manually.')

for code_, fp in COUNT_FILES.items():
    if not clinical.TCGA_CODE.eq(code_).any():
        print(f'{code_}: no missing patients, skip matrix inspection')
        continue
    sep = separator_for(fp)
    hdr = pd.read_csv(fp, sep=sep, nrows=0).columns.tolist()
    layout = detect_layout(fp, LAYOUT_OVERRIDE[code_])
    print(f'{code_}: {layout}, {len(hdr):,} source columns, first={hdr[0]!r}; first 3 headers={hdr[:3]}')

## Prepare and verify upload files

In [ ]:
def normalize_gene(g):
    g = str(g).strip()
    if not g or g.lower() in {'nan','none','na','gene_name','gene_id','symbol'}:
        return None
    if re.fullmatch(r'ENSG\d+(?:\.\d+)?',g,re.I):
        g = g.split('.')[0].upper()
    if g.startswith('__') or g.startswith('N_'):
        return None
    return g

def allow_sample(name, expected_patients):
    norm = normalize_barcode(name)
    if not norm or patient_id(norm) not in expected_patients:
        return False
    st = sample_type(norm)
    if st is not None and st != 1:
        return False
    if st is None and REQUIRE_EXPLICIT_TUMOR_SAMPLE_TYPE:
        return False
    return True

def read_cohort_counts(path, cohort, layout):
    sep = separator_for(path)
    hdr = pd.read_csv(path, sep=sep, nrows=0).columns.tolist()
    ids = set(clinical.loc[clinical.TCGA_CODE.eq(cohort),'patient_id'])
    if layout == 'genes_rows':
        gene_col = hdr[0]
        expr_cols = [c for c in hdr[1:] if allow_sample(c,ids)]
        if not expr_cols:
            raise ValueError(f'{cohort}: no expression columns matched clinical patients')
        genes, arrays = [], []
        for df in pd.read_csv(path, sep=sep, usecols=[gene_col]+expr_cols,
                              chunksize=COUNT_READ_CHUNK, low_memory=False):
            gen = df[gene_col].map(normalize_gene)
            mask = gen.notna()
            if not mask.any():
                continue
            arr = df.loc[mask,expr_cols].apply(pd.to_numeric,errors='coerce').to_numpy(dtype='float64').T
            arrays.append(arr)
            genes.extend(gen.loc[mask].tolist())
        if not arrays:
            raise ValueError(f'{cohort}: no valid gene counts')
        counts = pd.DataFrame(np.concatenate(arrays,axis=1),
                              index=[normalize_barcode(c) for c in expr_cols], columns=genes)
        del arrays
    else:
        sample_col = hdr[0]
        expr_cols = [c for c in hdr[1:] if c not in {'SAMPLE','TCGA_CODE','TISSUE','BATCH'}]
        kept=[]
        for df in pd.read_csv(path,sep=sep,chunksize=16,low_memory=False):
            norm = df[sample_col].map(normalize_barcode)
            mask = norm.map(lambda x: allow_sample(x,ids))
            if not mask.any():
                continue
            sub = df.loc[mask,expr_cols].apply(pd.to_numeric,errors='coerce')
            sub.index = norm[mask].tolist()
            kept.append(sub)
        if not kept:
            raise ValueError(f'{cohort}: no expression rows matched clinical patients')
        counts = pd.concat(kept,axis=0)
        keep_col = [normalize_gene(c) for c in counts.columns]
        valid = [i for i,g in enumerate(keep_col) if g is not None]
        counts = counts.iloc[:,valid].copy()
        counts.columns = [keep_col[i] for i in valid]
    if counts.index.duplicated().any():
        raise ValueError(f'{cohort}: duplicate expression sample names after normalization')
    duplicated_genes = int(counts.columns.duplicated().sum())
    if duplicated_genes:
        counts = counts.T.groupby(level=0,sort=False).sum().T
    if len(counts.columns) < MIN_GENES:
        raise ValueError(f'{cohort}: only {len(counts.columns)} genes; expected >= {MIN_GENES}')
    gene_ids = counts.columns.astype(str)
    n_genes_recognizable = sum(bool(re.fullmatch(r'ENSG\d+',g,re.I) or
                                  re.fullmatch(r'[A-Za-z][A-Za-z0-9_.\-]{1,40}',g)) for g in gene_ids)
    if n_genes_recognizable / max(1,len(gene_ids)) < 0.5:
        raise ValueError(f'{cohort}: most gene IDs are not recognizable symbols/Ensembl IDs; '
                         'check count.csv first column and gene annotation.')
    x = counts.to_numpy(dtype='float64', copy=False)
    if not np.isfinite(x).all() or (x < 0).any():
        raise ValueError(f'{cohort}: invalid, missing or negative counts')
    nonint = (np.abs(x - np.rint(x)) > 1e-4).mean()
    if nonint > 1e-4 and not ALLOW_FRACTIONAL_COUNTS:
        raise ValueError(f'{cohort}: {nonint:.1%} fractional values. Ensure input is raw gene counts, '
                         'not FPKM/TPM or transformed values.')
    lib = x.sum(axis=1)
    if (lib <= 0).any() or (x == 0).mean(axis=1).max() > 0.99:
        raise ValueError(f'{cohort}: zero-library or >99% zero-gene sample present')
    if np.median(lib) < 10000:
        warnings.warn(f'{cohort}: unusually low median library sum; inspect raw counts provenance')
    counts = counts.round().astype('int64') if not ALLOW_FRACTIONAL_COUNTS else counts.astype('float64')
    audit = pd.DataFrame({
        'expression_sample': counts.index.tolist(),
        'patient_id':[patient_id(v) for v in counts.index],
        'sample_type':[sample_type(v) for v in counts.index],
        'library_total':lib,
        'fraction_zero':(x == 0).mean(axis=1),
        'cohort_source':cohort,
    })
    print(f'{cohort}: kept {len(counts)} RNA samples, {len(counts.columns):,} genes, '
          f'merged {duplicated_genes} duplicate gene columns; median library {np.median(lib):,.0f}')
    return counts, audit

def balanced_slices(n, max_size, min_size):
    if n < min_size:
        raise ValueError(f'Too few samples ({n}) for CellHit; need >= {min_size}')
    parts = math.ceil(n/max_size)
    while parts > 1 and n // parts < min_size:
        parts -= 1
    sizes = [n//parts+int(k<n%parts) for k in range(parts)]
    if min(sizes) < min_size:
        raise ValueError('Could not create CellHit upload batches with >= 3 samples')
    offset=0
    for size in sizes:
        yield slice(offset,offset+size)
        offset += size

def verify_upload(path, n_expected, expected_cohort):
    head = pd.read_csv(path,nrows=0).columns.tolist()
    if head[:3] != ['SAMPLE','TCGA_CODE','TISSUE'] or len(head)-3 < MIN_GENES:
        raise ValueError(f'Bad web export header in {path}')
    seen=[]
    for chunk in pd.read_csv(path,chunksize=20):
        if not chunk['TCGA_CODE'].eq(expected_cohort).all():
            raise ValueError(f'Incorrect TCGA_CODE in {path}')
        if not chunk['TISSUE'].eq(TISSUE_BY_CODE[expected_cohort]).all():
            raise ValueError(f'Incorrect TISSUE in {path}')
        seen += chunk.SAMPLE.astype(str).tolist()
        a = chunk.iloc[:,3:].to_numpy(dtype='float64')
        if not np.isfinite(a).all() or (a<0).any():
            raise ValueError(f'Invalid output counts in {path}')
        if not ALLOW_FRACTIONAL_COUNTS and not np.allclose(a,np.rint(a)):
            raise ValueError(f'Unexpected fractional output counts in {path}')
    if len(seen) != n_expected or len(seen) != len(set(seen)):
        raise ValueError(f'Duplicate or missing output samples in {path}')
    return True

In [ ]:
manifest_rows=[]
selection_rows=[]
print('Building uploads for missing CellHit patients only...')

for stale in OUTPUT_DIR.glob('cellhit_upload_*_part*.csv.gz'):
    stale.unlink()

for code_, path in COUNT_FILES.items():
    if not clinical.TCGA_CODE.eq(code_).any():
        print(f'{code_}: no missing patients; skip reading entire count matrix')
        continue
    layout = detect_layout(path,LAYOUT_OVERRIDE[code_])
    count_matrix, audit = read_cohort_counts(path,code_,layout)
    group_ref = clinical.loc[clinical.TCGA_CODE.eq(code_), ['patient_id','clinical_sample','cancer','group_label']]
    audit = audit.merge(group_ref,on='patient_id',how='inner',validate='many_to_one')

    audit['primary_priority'] = audit['sample_type'].eq(1).astype('int8')
    audit = audit.sort_values(['patient_id','primary_priority','library_total','expression_sample'],
                              ascending=[True,False,False,True]).copy()
    audit['chosen'] = ~audit.patient_id.duplicated()
    audit.to_csv(OUTPUT_DIR / f'{code_.lower()}_expression_sample_selection_audit.csv',index=False)
    selected = audit.loc[audit.chosen].copy()
    selected['TCGA_CODE'] = code_
    selected['TISSUE'] = TISSUE_BY_CODE[code_]
    selection_rows.append(selected)
    if audit['sample_type'].isna().any():
        print(f'  NOTE {code_}: {audit.sample_type.isna().sum()} sample(s) lack an encoded TCGA sample-type. See audit.')
    if len(selected) < MIN_SAMPLES_PER_UPLOAD:
        warnings.warn(f'{code_}: only {len(selected)} matched patients, cannot create an upload with >=3 rows')
        del count_matrix
        continue

    expr = count_matrix.loc[selected['expression_sample'].tolist()].copy()
    expr.index.name = 'SAMPLE'
    metadata = selected.set_index('expression_sample').loc[expr.index,['TCGA_CODE','TISSUE']]
    upload = pd.concat([metadata,expr],axis=1).reset_index()
    if upload.columns[0] != 'SAMPLE':
        upload = upload.rename(columns={upload.columns[0]:'SAMPLE'})
    if upload.columns[:3].tolist() != ['SAMPLE','TCGA_CODE','TISSUE']:
        raise AssertionError('Web upload columns not in required order')
    for part, sl in enumerate(balanced_slices(len(upload),MAX_SAMPLES_PER_UPLOAD,MIN_SAMPLES_PER_UPLOAD),1):
        data = upload.iloc[sl]
        fp = OUTPUT_DIR / f'cellhit_upload_{code_.lower()}_part{part:02}.csv.gz'
        data.to_csv(fp,index=False,compression={'method':'gzip','compresslevel':5})
        verify_upload(fp,len(data),code_)
        manifest_rows.append({'upload_file':str(fp),'TCGA_CODE':code_,
                              'n_samples':len(data),'n_genes':len(data.columns)-3,
                              'size_MiB':round(fp.stat().st_size/1024**2,2),'verified':True})
        print(f'  wrote {fp.name}: {len(data)} tumors, {len(data.columns)-3:,} genes, '
              f'{fp.stat().st_size/1024**2:.2f} MiB')
    del count_matrix, expr, metadata, upload

if not selection_rows:
    raise ValueError('No expression samples matched any clinical cohort')
matched = pd.concat(selection_rows,ignore_index=True)
if matched.patient_id.duplicated().any():
    raise ValueError('Duplicate patient IDs across exported cohorts')
matched.to_csv(OUTPUT_DIR/'matched_samples_with_msi_tmb.csv',index=False)
manifest=pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR/'cellhit_upload_file_manifest.csv',index=False)
ref = clinical_full.groupby(['TCGA_CODE','group_label']).size().rename('all_clinical_patients')
needed = clinical.groupby(['TCGA_CODE','group_label']).size().rename('missing_cellhit_patients')
observed = matched.groupby(['TCGA_CODE','group_label']).size().rename('matched_missing_patients_with_raw_counts')
coverage = pd.concat([ref,needed,observed],axis=1).fillna(0).reset_index()
for col in ['all_clinical_patients','missing_cellhit_patients','matched_missing_patients_with_raw_counts']:
    coverage[col] = coverage[col].astype('int64')
coverage['fraction_missing_recovered'] = (coverage.matched_missing_patients_with_raw_counts /
                                          coverage.missing_cellhit_patients.replace(0,np.nan))
coverage.to_csv(OUTPUT_DIR/'clinical_to_raw_counts_coverage.csv',index=False)
unrecovered = clinical.loc[~clinical.patient_id.isin(matched.patient_id)].copy()
unrecovered.to_csv(OUTPUT_DIR/'missing_cellhit_but_no_raw_counts.csv',index=False)
print('\nClinical coverage:')
display(coverage)
print('\nUpload manifest:')
display(manifest)
print('Saved matching table:',OUTPUT_DIR/'matched_samples_with_msi_tmb.csv')

## CellHit web export

Submit the generated count matrices to CellHit and save the raw prediction CSV files in `data/CellHit/results/`.

In [ ]:
print('Files to upload to CellHit:')
for fp in manifest['upload_file']:
    print(' ', fp)
print('\nClinical coverage and upload manifest:')
for name in ['matched_samples_with_msi_tmb.csv','clinical_to_raw_counts_coverage.csv',
             'cellhit_upload_file_manifest.csv']:
    print(' ',OUTPUT_DIR/name)
print('Already covered versus missing:')
display(summary)
print('Not exportable due to absent raw counts:',len(unrecovered))